# GPU Dask Cluster Test

This notebook verifies that a Dask cluster can be created with GPU resources and that GPU computations can be executed successfully.


In [ ]:
# Imports
import sys
from dask_kubernetes.operator import KubeCluster
from dask import delayed
import dask.array as da
from dask.distributed import PipInstall, CondaInstall

import numpy as np

In [ ]:
ENABLE_GPU = False # Changee to True to use GPU workers
PYTHON_VERSION = sys.version_info.minor
NAMESPACE="ws-<name>" # Change to your namespace, where <name> is your workspace name

In [ ]:
def get_config():
    image_domain = "public.ecr.aws/eodh"
    image = "eodh-default-notebook:python-3.13-2026.02.3" if PYTHON_VERSION == 13 else "eodh-default-notebook:python-3.12-2026.02.3"
    if ENABLE_GPU:
        image = "eodh-default-notebook:python-3.13-2026.03.1-cuda" if PYTHON_VERSION == 13 else "eodh-default-notebook:python-3.12-2026.03.1-cuda"
    limits = {"memory": "4Gi", "cpu": "2"}
    print("Using notebook image:", image)
    return {
        "resources": {
            "requests": {"memory": "2Gi", "cpu": "1"},
            "limits": limits if not ENABLE_GPU else limits | {"nvidia.com/gpu": "1"}
        },
        "n_workers": 2,
        "image": f"{image_domain}/{image}"
    }

In [ ]:
# Create the Dask cluster with GPU resources
cluster = KubeCluster(
    name="test-gpu",
    namespace=NAMESPACE,
    **get_config()
)
cluster
### FOR GPUS, this step will take a more than 5 minutes to provision GPU nodes.

In [ ]:
client = cluster.get_client()
client

In [ ]:
def gpu_computation(x):
    print("Running on Python:", sys.version)
    if ENABLE_GPU:
        """Simple GPU computation to test GPU availability."""
        import cupy as cp
        # Create a GPU array and perform operations
        gpu_array = cp.asarray(x)
        result = cp.sum(gpu_array ** 2)
        return float(result)
    else:
        # Create a CPU array and perform operations
        import numpy as np
        return float(np.sum(x ** 2))

In [ ]:
# Create GPU computation tasks
sample_size = 400
test_data = [np.random.rand(1000) for _ in range(sample_size)]

# Create delayed tasks
tasks = [delayed(gpu_computation)(data) for data in test_data]

# Compute results
results = client.compute(tasks, sync=True)

compute_type = "GPU" if ENABLE_GPU else "CPU"

print(f"{compute_type} Computation Results: {results[0:10]}")
print(f"Number of successful computations: {len(results)}")

assert len(results) == sample_size, "Expected results from computation"
assert all(isinstance(r, float) for r in results), "All results should be floats"

print(f"✓ {compute_type} cluster test passed!")

In [ ]:
client.close()
cluster.close()